# Joan Tryhard

### Imports

In [66]:
import pandas as pd
import sklearn
import imblearn

### Get Data and Preprocess

In [67]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd

TESTING_WITH_TRAIN_DATA = False


train = pd.read_csv("data/processed_binary_train.csv")
train.drop(axis=1, columns=['eigenvector', 'Unnamed: 0'])

test = pd.read_csv("data/processed_binary_test.csv")
test.drop(axis=1, columns=['eigenvector', 'Unnamed: 0'])


if TESTING_WITH_TRAIN_DATA:

    # Split into train and test sets
    sentences = train[['language', 'sentence_id']].drop_duplicates()
    sentences_train, sentences_test = train_test_split(sentences,test_size=0.2,random_state=42)
    train_set = pd.merge(train, sentences_train, on=['language', 'sentence_id'])
    test_set = pd.merge(train, sentences_test, on=['language', 'sentence_id'])

    # One-hot encode 'language' in train and test sets
    enc = OneHotEncoder(sparse_output=True)
    language_encoded = enc.fit_transform(train_set[['language']])
    language_df = pd.DataFrame(language_encoded.toarray(),
                            columns=enc.get_feature_names_out(['language']),
                            index=train_set.index)
    train = pd.concat([train_set.drop(columns=['language']), language_df], axis=1)

    # Same for test set
    language_encoded_test = enc.transform(test_set[['language']])
    language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                    columns=enc.get_feature_names_out(['language']),
                                    index=test_set.index)
    test = pd.concat([test_set.drop(columns=['language']), language_df_test], axis=1)

    # Prepare data and labels
    X_train = train.drop(columns=['root'])
    y_train = train['root']
    X_test = test.drop(columns=['root'])
    y_test = test['root']

else: 
    # One-hot encode 'language' in train and test (all test data) sets
    enc = OneHotEncoder(sparse_output=True)
    language_encoded = enc.fit_transform(train[['language']])
    language_df = pd.DataFrame(language_encoded.toarray(),
                            columns=enc.get_feature_names_out(['language']),
                            index=train.index)
    train = pd.concat([train.drop(columns=['language']), language_df], axis=1)

    # Same for test set
    language_encoded_test = enc.transform(test[['language']])
    language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                    columns=enc.get_feature_names_out(['language']),
                                    index=test.index)
    test = pd.concat([test.drop(columns=['language']), language_df_test], axis=1)

    #Prepare data and labels
    X_train = train.drop(columns=['root'])
    y_train = train['root']
    X_test = test
    # there is no y_test, it is what we want to predict!

## Models

### Unimodel Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# import linear classifier
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

RandomForestClassifier()

In [69]:
prob_predictions = clf.predict_proba(X_test)
# convert to 0s and 1s integers
predictions = []
for i in range(len(prob_predictions)):
    if prob_predictions[i][1] > 0.5:
        predictions.append(1)
    else:
        predictions.append(0)

if TESTING_WITH_TRAIN_DATA:

    y_test_labels = y_test.values

    len(predictions), len(y_test_labels)

    # get the error on the test set with different metrics
    from sklearn.metrics import classification_report, confusion_matrix
    print(confusion_matrix(y_test_labels, predictions))
    print(classification_report(y_test_labels, predictions, target_names=['0', '1']))

In [76]:
# add predictions to the test set
X_test['root'] = predictions

# Build final predictions
rows = []
i = 1
for _, row in X_test.iterrows():
    if row['root'] == 1:
        node_id = row['vertex_id']
        rows.append({'id': i, 'root': int(node_id)})
        i += 1

# Create DataFrame from collected rows
final_predictions = pd.DataFrame(rows)
final_predictions.to_csv('data/predictions.csv', index=False)

In [77]:
X_test

,Unnamed: 0,sentence_id,n,vertex_id,in_degree,out_degree,degree,is_leaf,descendants,closeness,...,language_Japanese,language_Korean,language_Polish,language_Portuguese,language_Russian,language_Spanish,language_Swedish,language_Thai,language_Turkish,root
0,0,1,43,38,1,1,2,0,1,0.044218,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1,1,43,33,1,0,1,1,0,0.044444,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2,1,43,10,1,3,4,0,8,0.043956,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,3,1,43,24,1,1,2,0,1,0.044818,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,4,1,43,16,1,0,1,1,0,0.044974,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194643,194643,993,16,7,1,1,2,0,2,0.100000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
194644,194644,993,16,5,1,1,2,0,1,0.106667,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
194645,194645,993,16,11,1,0,1,1,0,0.111111,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
194646,194646,993,16,2,1,1,2,0,1,0.100000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
